# Actividad 16. Análisis de desempeño de algoritmo paralelo

**Materia:** Lenguajes de Computación · Otoño 2026

| Nombre completo | No. de cuenta |
|---|---|
| Angel Rugerio Jiménez | 201720 |

**Objetivo:** ejecutar la simulación meteorológica de la clase con $P \in \{1, 2, 4, 8\}$ procesos, medir el tiempo total y calcular la aceleración $S_P = T_1 / T_P$ y la eficiencia $E_P = S_P / P$.

## Libreta base

El código de las cuatro fases está en `simulacion.py`. Es el de la libreta de la clase con las correcciones necesarias para que corra:

- **Particionamiento:** cada proceso recibe su número (`rank`) para inicializar su propia franja; antes todos generaban la misma. Si $N$ no es divisible entre $P$, las filas sobrantes se reparten.
- **Comunicación:** las condiciones de frontera se aplican cuando **no** hay vecino (`grid[0, :] = grid[1, :]`).
- **Cómputo:** los cuatro vecinos usan rebanadas del mismo tamaño que la región interior.
- **Integración:** `historico.append((step, ...))` guarda una tupla, y `ejecutar_simulacion` crea los procesos, los pipes y mide el tiempo.

Las funciones viven en un módulo porque Python 3.14 arranca los procesos con `forkserver`, que no puede ejecutar funciones definidas dentro de la libreta. Se verificó que la cuadrícula final es la misma con 1, 2, 4 y 8 procesos.

In [1]:
import platform
import statistics

from simulacion import ejecutar_simulacion

PROCESOS = [1, 2, 4, 8]
PASOS = 1000
REPETICIONES = 5

ejecutar_simulacion(2, 100, 10)  # calentamiento: arranca el servidor de procesos

def medir(N):
    tiempos = {}
    for P in PROCESOS:
        tiempos[P] = statistics.median(ejecutar_simulacion(P, N, PASOS)[0] for _ in range(REPETICIONES))
    print(f"N = {N}, {PASOS} pasos, mediana de {REPETICIONES} ejecuciones")
    print(f"| Procesos (P) | Tiempo total T_P (s) | Aceleración S_P | Eficiencia E_P |")
    print(f"| --- | --- | --- | --- |")
    for P in PROCESOS:
        S = tiempos[1] / tiempos[P]
        print(f"| {P} | {tiempos[P]:.3f} | {S:.2f} | {S / P:.0%} |")
    return tiempos

print(platform.processor() or platform.machine(), "| Python", platform.python_version())

x86_64 | Python 3.14.4


## Evaluación con $N = 100$

In [2]:
tiempos_100 = medir(100)

N = 100, 1000 pasos, mediana de 5 ejecuciones
| Procesos (P) | Tiempo total T_P (s) | Aceleración S_P | Eficiencia E_P |
| --- | --- | --- | --- |
| 1 | 0.187 | 1.00 | 100% |
| 2 | 0.192 | 0.98 | 49% |
| 4 | 0.250 | 0.75 | 19% |
| 8 | 0.403 | 0.47 | 6% |


## Evaluación con $N = 500$

In [3]:
tiempos_500 = medir(500)

N = 500, 1000 pasos, mediana de 5 ejecuciones
| Procesos (P) | Tiempo total T_P (s) | Aceleración S_P | Eficiencia E_P |
| --- | --- | --- | --- |
| 1 | 1.778 | 1.00 | 100% |
| 2 | 1.100 | 1.62 | 81% |
| 4 | 0.840 | 2.12 | 53% |
| 8 | 1.004 | 1.77 | 22% |


## Resultados

Medido el 2026-11-11 en un Intel Core Ultra 5 228V (8 núcleos), Linux, Python 3.14.4, con 1000 pasos de tiempo. Cada tiempo es la mediana de 5 ejecuciones e incluye crear los procesos, simular y recibir el histórico.

| Procesos (P) | $T_P$ con $N = 100$ (s) | $S_P$ | $E_P$ | $T_P$ con $N = 500$ (s) | $S_P$ | $E_P$ |
| --- | --- | --- | --- | --- | --- | --- |
| 1 (secuencial) | 0.187 | 1.00 | 100 % | 1.778 | 1.00 | 100 % |
| 2 | 0.192 | 0.98 | 49 % | 1.100 | 1.62 | 81 % |
| 4 | 0.250 | 0.75 | 19 % | 0.840 | 2.12 | 53 % |
| 8 | 0.403 | 0.47 | 6 % | 1.004 | 1.77 | 22 % |

Con $N = 100$ cada proceso tiene tan poco cálculo que crear procesos e intercambiar bordes cuesta más de lo que se ahorra: el programa es **más lento** al paralelizarlo. Con $N = 500$ sí hay aceleración, con el mejor resultado en $P = 4$. Con $P = 8$ empeora: hay más procesos que crear y más bordes que intercambiar, y probablemente influye que 4 de los 8 núcleos del procesador son de bajo consumo.

## Preguntas

**1. ¿Cómo afecta el tamaño de la franja de Ghost Cells enviada por los pipes al tiempo total de ejecución cuando se incrementa la resolución de la malla a $N = 500$?**

En cada paso, cada proceso envía a lo mucho dos filas de $N + 2$ valores: con $N = 100$ son $102 \times 8 = 816$ bytes por fila y con $N = 500$ son $502 \times 8 = 4016$ bytes, unas **5 veces más**. Pero el cálculo de cada proceso crece con el área de su franja, $N^2 / P$ celdas, es decir **25 veces más**. La comunicación crece con el borde y el cálculo con el área, así que al aumentar $N$ la comunicación pesa proporcionalmente menos. Por eso con $N = 100$ la aceleración con 8 procesos es 0.47 y con $N = 500$ sube a 1.77. Aun así, la franja sigue costando: cada paso exige enviar y esperar los bordes de los vecinos, y ese costo crece con $P$, porque hay más fronteras.

**2. Si una tormenta severa genera mayor carga de cálculo numérico en un subdominio específico de la cuadrícula, ¿por qué la asignación estática de tareas deja de ser eficiente y qué alternativa de balanceo dinámico de carga debería implementarse?**

Con asignación estática, cada proceso tiene una franja fija desde el inicio. Si la tormenta cae en una franja, ese proceso tarda más en cada paso, y como todos tienen que intercambiar bordes antes del siguiente paso, los demás se quedan **esperando** en `recv()`. El tiempo total queda dictado por el proceso más cargado y la eficiencia baja, aunque las franjas tengan el mismo número de filas.

La alternativa es un **balanceo semi-estático o dinámico**: medir cada cierto número de pasos cuánto tarda cada proceso y **mover las fronteras**, dando menos filas a la franja de la tormenta y más a las demás. Otra opción es dividir la malla en muchas franjas pequeñas y que un proceso maestro las reparta conforme los workers terminan (modelo Master/Worker). En el código de esta actividad todas las celdas cuestan lo mismo, así que este efecto no aparece en las mediciones.

**3. Si la fase de comunicación e intercambio de bordes representa un 10 % del tiempo total inalterable en paralelo, ¿cuál es la aceleración máxima teórica alcanzable según la Ley de Amdahl sin importar el número de procesadores añadidos?**

Con una fracción secuencial de $0.1$ y una fracción paralelizable de $0.9$:

$$S(P) = \frac{1}{0.1 + \dfrac{0.9}{P}} \qquad \lim_{P \to \infty} S(P) = \frac{1}{0.1} = 10$$

La aceleración máxima es de **10 veces**, sin importar cuántos procesadores se agreguen. Con los 8 procesos de esta actividad, la aceleración máxima sería $1 / (0.1 + 0.1125) \approx 4.71$.